# Impedance ML Engineering — Session 19
## Capstone: design and run a controlled development benchmark
**Core: 60 minutes · Capstone · Optional extension: 20–40 minutes**

## Why this matters
Which representation and model would you choose for next-visit prediction, given accuracy, evidence, and engineering cost?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Prespecify a benchmark contract and bounded search.
- Compare baseline, statistical/tabular, and neural approaches fairly.
- Use development-only selection and controlled ablations.
- Freeze a candidate before final evaluation.

## R-to-Python / prerequisite bridge
Combine familiar building blocks; the challenge is experimental judgment and reproducibility rather than a new API.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
A benchmark compares complete learning procedures, including preprocessing, tuning, early stopping, input availability, and computational budget. Changing any of these can alter what a score means. A prespecified rule protects against choosing whatever looks best after repeated inspection.

The core capstone reuses fixed recipes and the same validation patients; it is a controlled development comparison, not independent evidence. The untouched patient test supports a final frozen-model comparison, while new real-world performance still needs external evidence. If development refitting is chosen, freeze all architecture/hyperparameters and the neural epoch count first.

Record seeds, package/data versions, prediction tables, feature order, parameters, timing, and the signed difference against persistence. A simpler model can be selected for stability and inference cost even with a small validation disadvantage.

## Task 1 — Write the benchmark contract (5 min)
Use the original unseen-patient next-visit continuous target and fixed 24-patient test reserve. Declare primary patient-average MAE and secondary pair MAE, horizons, available features, and maximum budget. State how selection will account for complexity. Predict the independent development sample size (72 patients).

## Task 2 — Choose one scientific feature hypothesis (6 min)
Select a prespecified spatial contrast or asymmetry feature and document QC/missingness/units. State the statistical intuition from Sessions 04–05. Keep association evidence separate from causal interpretation. Design one ablation rather than searching every feature subset.

## Task 3 — Specify three trained candidates (6 min)
Core candidates: summary/engineered ridge, one selected tree/boosting recipe, and one small MLP OR CNN/fusion recipe. Include persistence as a nonfitted reference. Reuse earlier recipes and cap the neural budget at 60 epochs per run. Comparing both MLP and CNN is an optional extension.

## Task 4 — Align selection evaluation (10 min)
Use the original 54/18 train/validation split as the core selection design for all candidates. If using CV, every model must share identical patient-grouped outer folds and neural early stopping must use an inner patient split only. Document reused-validation limitations. Do not rank earlier scores from incompatible splits.

## Task 5 — Run the bounded benchmark (12 min)
Fit each candidate with preprocessing learned only in its permitted training data. Save validation predictions with IDs, feature names, parameters, seed, elapsed time, and best neural epoch. Assert row alignment, finite predictions, and identical target rows. Use fixed recipes rather than launching another broad search.

## Task 6 — Interpret the controlled ablation (7 min)
Compare your feature hypothesis included versus excluded for one reference model. Report validation MAE difference and subgroup counts. Label it exploratory if the same validation set guided its design. Explain whether increased predictive value justifies extra data preparation.

## Task 7 — Freeze the final candidate (6 min)
Apply your documented selection rule. Save candidate identity, preprocessing, configuration, and fixed neural epoch count before viewing test. For the optional development refit, train on all 72 development patients with frozen hyperparameters and that fixed epoch count; do not early-stop on test. Simpler alternative: evaluate the restored training/validation-selected checkpoint directly and state its training size.

## Task 8 — Complete the pre-evaluation audit (0–2 min)
Check test patients have never influenced learned preprocessing, early stopping, features, or hyperparameters. Document unavoidable course-wide validation reuse. If test results have already been inspected, stop calling it untouched: declare exploratory reuse and request an external dataset for credible evaluation. Write exactly which scores Session 20 will report.

## Retrieval practice
From memory, draw the data flow from raw current matrices to a frozen inference artifact, identifying every learned transformation.

## Professional reasoning
1. What could go wrong if neural models use one convenient split while trees use CV?
2. Why does more tuning budget make a benchmark unfair?
3. How would you justify selecting a simpler model with slightly higher validation MAE?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Run identical three-fold grouped outer CV for the three trained candidates, with inner grouped early stopping for neural models. Allow several additional sessions; do not squeeze this into the core hour.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)